# Demanda eléctrica diaria en España: forecasting

Estimamos la demanda eléctrica diaria de la península en GWh con el historial, el calendario y, opcionalmente, la temperatura media de Madrid. La demanda procede de la API pública de Red Eléctrica y el tiempo de Open-Meteo.

**En sencillo:** el modelo busca patrones en el consumo pasado para estimar el de días futuros. No mezclamos los días: probamos con las fechas más recientes, como ocurriría al usar el modelo para pronosticar.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from ml_al_alfa.datasets import FORECAST_CONFIG, load_forecast_frame
from ml_al_alfa.forecasting import forecast_to_date, train_forecast_and_save
from ml_al_alfa.train import ARTIFACTS_DIR, RAW_DIR

sns.set_theme(style="whitegrid")
DATASET = "demand"
CONFIG = FORECAST_CONFIG[DATASET]
TARGET = CONFIG["target"]
WEATHER = CONFIG["exogenous"][0]

## 1. Obtener y revisar los datos

Si no existe un CSV preparado en `data/raw/`, el proyecto combina la serie diaria de demanda de Red Eléctrica con el tiempo histórico de Madrid descargado desde Open-Meteo. Se guardan los datos combinados localmente para reutilizarlos.

In [ ]:
series = load_forecast_frame(DATASET, RAW_DIR)
series["date"] = pd.to_datetime(series["date"])
series = series.sort_values("date").reset_index(drop=True)
print(f"Filas: {len(series):,}; fechas: {series['date'].min().date()} – {series['date'].max().date()}")
print(f"Valores ausentes: {series[[TARGET, WEATHER]].isna().sum().sum()}")
display(series.head())
display(series[[TARGET, WEATHER]].describe().T)

## 2. Explorar consumo y temperatura

La demanda muestra patrones semanales y estacionales. La temperatura aquí corresponde a Madrid: es una característica meteorológica local, no la temperatura media de toda España.

In [ ]:
series["día de la semana"] = series["date"].dt.day_name()
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
fig, axes = plt.subplots(3, 1, figsize=(14, 12))
axes[0].plot(series["date"], series[TARGET], linewidth=0.8)
axes[0].set(title="Demanda eléctrica diaria", xlabel="Fecha", ylabel="GWh")
axes[1].plot(series["date"], series[WEATHER], color="darkorange", linewidth=0.8)
axes[1].set(title="Temperatura media diaria en Madrid", xlabel="Fecha", ylabel="°C")
sns.boxplot(data=series, x="día de la semana", y=TARGET, order=weekday_order, ax=axes[2])
axes[2].tick_params(axis="x", rotation=25)
axes[2].set(title="Demanda por día de la semana", xlabel="Día", ylabel="GWh")
plt.tight_layout()
plt.show()

sns.scatterplot(data=series, x=WEATHER, y=TARGET, alpha=0.35)
plt.title("Demanda diaria y temperatura media de Madrid")
plt.xlabel("Temperatura (°C)")
plt.ylabel("Demanda (GWh)")
plt.show()

## 3. Entrenar sin barajar las fechas

El baseline predice el consumo de hace siete días. Comparamos regresión lineal y Random Forest, primero con calendario e historial y después añadiendo la temperatura observada. La temperatura del test sirve para medir el efecto de esa variable, pero para un pronóstico futuro habrá que proporcionar una temperatura estimada.

In [ ]:
bundle = train_forecast_and_save(
    series[["date", TARGET, WEATHER]],
    dataset_name=DATASET,
    output_path=ARTIFACTS_DIR / "demand_model.joblib",
)
metrics = pd.DataFrame(bundle["metrics"]).T.sort_values("mae")
display(metrics.rename(columns={"mae": "MAE (GWh/día, menor es mejor)", "r2": "R²"}))
print(f"Inicio del test cronológico: {bundle['forecast']['split_date']}")
print(f"Mejor variante: {bundle['forecast']['best_variant']}")
print(f"Mejor método: {bundle['model_name']}")
print(f"Modelo para Streamlit: {ARTIFACTS_DIR / 'demand_model.joblib'}")

## 4. Revisar el test

Comparamos los valores reales con la predicción del mejor modelo en las últimas fechas que se reservaron.

In [ ]:
evaluation = pd.DataFrame(bundle["forecast"]["evaluation"])
evaluation["date"] = pd.to_datetime(evaluation["dates"])
display(evaluation[["date", "actual", "predicted"]].tail(10))
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(evaluation["date"], evaluation["actual"], label="Real", linewidth=1)
ax.plot(evaluation["date"], evaluation["predicted"], label="Pronóstico", linewidth=1)
ax.set(title="Pronósticos frente a demanda real", xlabel="Fecha", ylabel="Demanda (GWh)")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Pronosticar una fecha futura

Para el ejemplo usamos la temperatura media histórica como supuesto. No es una predicción meteorológica. Cambia el valor y el horizonte para probar otros escenarios.

In [ ]:
weather_assumption = {WEATHER: float(series[WEATHER].median())}
future = forecast_to_date(
    bundle,
    variant_label=bundle["forecast"]["best_variant"],
    days_ahead=7,
    exogenous_values=weather_assumption,
)
future_frame = pd.DataFrame(future, columns=["date", "estimated_demand_gwh"])
display(future_frame)
print(f"Supuesto de temperatura diaria: {weather_assumption[WEATHER]:.1f} °C")
print("En Streamlit introduce la temperatura que esperas para el horizonte elegido.")